Import drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

Data

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Thesis docs/tweet_lyrics_vad.csv')

df.head()

,Unnamed: 0,title,artist,disorder,sentiment_direction,sentiment_score,lyric,emotion_anger_score,emotion_disgust_score,emotion_fear_score,...,mean_dominance_VAD,mean_valence_VAD,tweet_model_text,lyric_model_text,tweet_valence,tweet_arousal,tweet_dominance,lyric_valence,lyric_arousal,lyric_dominance
0,0,Burnin Bridges / Long Day (feat. IDK),Quadeca,depression,POSITIVE,0.9971,Highest To Lowest: Quadeca LyricsQuadeca's Son...,0.0294,0.0014,0.0132,...,0.075335,0.055654,"18. He/They. Bisexual. Socialist. Filmmaker, m...",Highest To Lowest: Quadeca LyricsQuadeca's Son...,2.998211,2.962549,3.045191,3.146908,3.134274,3.088945
1,1,She's A Lady,Tom Jones,control,POSITIVE,0.9988,She’s a Lady Lyrics[Verse 1]\nWell she's all y...,0.0271,0.0203,0.0071,...,0.140663,0.105500,One song everyday (hopefully) of 2020.,She’s a Lady Lyrics[Verse 1]\nWell she's all y...,3.256605,2.998280,3.037257,3.211000,3.499823,3.281325
2,2,Lilies of the Valley,David Byrne,control,POSITIVE,0.9532,Lilies of the Valley Lyrics[Verse 1]\nMomma sh...,0.0252,0.0054,0.0556,...,0.093785,-0.189674,One song everyday (hopefully) of 2020.,Lilies of the Valley Lyrics[Verse 1]\nMomma sh...,3.256605,2.998280,3.037257,2.620652,3.451024,3.187569
3,3,School's Out,Alice Cooper,control,NEGATIVE,0.9995,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",0.1258,0.1330,0.0149,...,0.061629,-0.165449,One song everyday (hopefully) of 2020.,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",3.256605,2.998280,3.037257,2.669102,3.398807,3.123258
4,4,Call My Friends,Shawn Mendes,depression,POSITIVE,0.9978,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",0.0016,0.0006,0.0015,...,0.082034,0.017132,ariana & shawn + tom | he/him | 19,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",3.053620,2.922463,3.139341,3.034263,3.343280,3.164068


Prep

In [5]:
MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)

model.eval()

print("Using device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

Dep and C

In [6]:
target_groups ={"depression", "control"}
mask = df["disorder"].astype(str).str.strip().str.lower().isin(target_groups)

texts = df.loc[mask, "tweet_model_text"].fillna("").astype(str).tolist()

Get emotions from the model config

In [7]:
id2label = model.config.id2label
emotion_labels = [id2label[i] for i in range(len(id2label))]

Settings

In [8]:
def predict_emotions_batch(texts, batch_size=BATCH_SIZE):
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring tweets"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            max_length=512,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)

        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)

Train (1h)

In [9]:
emotion_probs = predict_emotions_batch(texts)

Scoring tweets:   0%|          | 0/1743 [00:00<?, ?it/s]

One column per emotion + a top-emotion label column

In [10]:
score_cols = [f'tweet_emotion_{label}_score' for label in emotion_labels]

for idx, col in enumerate(score_cols):
    df.loc[mask, col] = emotion_probs[:, idx]

df.loc[mask, "tweet_top_emotion"] = (
    df.loc[mask, score_cols]
    .idxmax(axis=1)
    .str.replace("tweet_emotion_", "", regex=False)
    .str.replace("_score", "", regex=False)
)

df.loc[mask, ['disorder', 'tweet_model_text', 'tweet_top_emotion'] + score_cols].head()

,disorder,tweet_model_text,tweet_top_emotion,tweet_emotion_anger_score,tweet_emotion_disgust_score,tweet_emotion_fear_score,tweet_emotion_joy_score,tweet_emotion_neutral_score,tweet_emotion_sadness_score,tweet_emotion_surprise_score
0,depression,"18. He/They. Bisexual. Socialist. Filmmaker, m...",neutral,0.008775,0.076281,0.008152,0.017913,0.777388,0.087682,0.023809
1,control,One song everyday (hopefully) of 2020.,neutral,0.004016,0.002873,0.002972,0.045880,0.883454,0.003640,0.057165
2,control,One song everyday (hopefully) of 2020.,neutral,0.004016,0.002873,0.002972,0.045880,0.883454,0.003640,0.057165
3,control,One song everyday (hopefully) of 2020.,neutral,0.004016,0.002873,0.002972,0.045880,0.883454,0.003640,0.057165
4,depression,ariana & shawn + tom | he/him | 19,neutral,0.006954,0.002324,0.002624,0.113305,0.424361,0.099336,0.351096


Shave

In [11]:
df.to_csv('/content/drive/MyDrive/Thesis docs/tweet_lyrics_vad_go.csv', index=False)